# Spectra — quantified modal / epistemic planner
### Colab quick-start (Python 3.13)

This notebook installs the `shadowprover` package, builds the higher-order E-prover it depends on, and runs an epistemic planning example (the "grapevine" problem).

**Requirements**
1. A Colab runtime on **Python 3.13** (`!python --version`). The wheel is version-locked to 3.13.
2. Higher-order E-prover (built below).

## 1. Check the Python version
The wheel only runs on Python 3.13 — confirm the runtime before installing.

In [1]:
!python --version   # expect Python 3.13.x

Python 3.13.18


## 2. Install shadowprover (from S3)

In [2]:
!pip install https://shadowprover-spectra.s3.amazonaws.com/py313/shadowprover-1.2.59-py3-none-any.whl
!pip install python-Levenshtein

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 14.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [shadowprover]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 69.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [python-Levenshtein]


## 3. Build higher-order E-prover
Clone and build E-prover with higher-order support. `make` takes a few minutes.

In [3]:
!git clone https://github.com/eprover/eprover
!cd eprover && ./configure --enable-ho && make rebuild
!./eprover/PROVER/eprover-ho --version

Cloning into 'eprover'...
remote: Enumerating objects: 36464, done.
remote: Counting objects: 100% (3557/3557), done.
remote: Compressing objects: 100% (535/535), done.
remote: Total 36464 (delta 3308), reused 3040 (delta 3022), pack-reused 32907 (from 3)
Receiving objects: 100% (36464/36464), 155.23 MiB | 17.01 MiB/s, done.
Resolving deltas: 100% (29925/29925), done.
Configuring with executable path /content/eprover/PROVER
Configuring with man path /content/eprover/DOC/man
echo 'Configuring build system'
Configuring build system
make       links
make[1]: Entering directory '/content/eprover'
make[1]: Leaving directory '/content/eprover'
make       depend
make[1]: Entering directory '/content/eprover'
make[2]: Entering directory '/content/eprover/CONTRIB'
make[2]: Nothing to be done for 'depend'.
make[2]: Leaving directory '/content/eprover/CONTRIB'
make[2]: Entering directory '/content/eprover/BASICS'
gcc -M -O03 -fno-common -fomit-frame-pointer  -Wall    -DNDEBUG -DFAST_EXIT  -DNUMAR

## 4. Point shadowprover at E-prover
`EPROVER_HOME` must be the eprover **root** directory (shadowprover looks for `$EPROVER_HOME/PROVER/eprover-ho`).

In [4]:
import os
os.environ['EPROVER_HOME'] = '/content/eprover/'

## 5. Smoke test
A trivial modus-ponens proof to confirm the install + E-prover work end to end.

In [5]:
from shadowprover.syntax.reader import r
from shadowprover.fol.fol_prover import fol_prove
print('provable  ->', fol_prove({r('(if (Human socrates) (Mortal socrates))'), r('(Human socrates)')}, r('(Mortal socrates)'))[0])
print('unprovable->', fol_prove({r('(Human socrates)')}, r('(Mortal socrates)'))[0])

provable  -> True
unprovable-> False


## 6. Imports

In [6]:
import shadowprover
from shadowprover.syntax.reader import r
from shadowprover.inference_systems.reader import read_inference_system
from shadowprover.inference_systems.cognitive_calculus import cc_prover
from shadowprover.reasoners.shadow_prover import shadow_prover as sprover
from shadowprover.reasoners.planner import run_spectra, Action
from edn_format import loads

## 7. Define the planning domain

In [7]:
domain = {r("a"), r("b"), r("c"), r("p1"), r("p2")}

In [8]:
background = set(list(map(r, [
        "(agent a)",
        "(agent b)",
        "(agent c)",
        "(room p1)",
        "(room p2)",

        "(not (= a b))",
        "(not (= a c))",
        "(not (= b c))"

        ])))

In [9]:
actions = [
        Action(r("(left ?a)"),
            precondition=r("(and (agent ?a) (at ?a p2))"),
            additions={r("(at ?a p1)"), r("(not (at ?a p2))")},
            deletions={r("(at ?a p2)"), r("(not (at ?a p1))")},
        ),
        Action(r("(right ?a)"),
            precondition=r("(and (agent ?a) (at ?a p1))"),
            additions={r("(at ?a p2)"), r("(not (at ?a p1))")},
            deletions={r("(at  ?a p1)"), r("(not (at ?a p2))")},
        ),
        Action(r("(shareboth ?a1 ?a2 ?a3 ?r)"),
            precondition= r(
                """(and
                    (agent ?a1)
                    (agent ?a2)
                    (agent ?a3)
                    (room ?r)
                    ; Precondition
                    (at ?a1 ?r)
                    (at ?a2 ?r)
                    (at ?a3 ?r)
                    (not (= ?a1 ?a2))
                    (not (= ?a1 ?a3))
                    (not (= ?a2 ?a3))
            )"""),
            additions={
                r("(Believes! ?a2 (the ?a1))"),
                r("(Believes! ?a3 (the ?a1))"),
                r("(Believes! ?a1 (Believes! ?a2 (the ?a1)))"),
                r("(Believes! ?a1 (Believes! ?a3 (the ?a1)))"),

            },
            deletions={
                r("(not (Believes! ?a2 (the ?a1)))"),
                r("(not (Believes! ?a3 (the ?a1)))"),
                r("(not (Believes! ?a1 (Believes! ?a2 (the ?a1))))"),
                r("(not (Believes! ?a1 (Believes! ?a3 (the ?a1))))")

            },
        ),

        Action(r("(sharesingle ?a1 ?a2 ?a3 ?r)"),
            precondition= r(
                """(and
                    (agent ?a1)
                    (agent ?a2)
                    (agent ?a3)
                    (room ?r)
                    ; Precondition
                    (at ?a1 ?r)
                    (at ?a2 ?r)
                    (not (at ?a3 ?r))
                    (not (= ?a1 ?a2))
                    (not (= ?a1 ?a3))
                    (not (= ?a2 ?a3))
            )"""),
            additions={
                r("(Believes! ?a2 (the ?a1))"),
                r("(Believes! ?a1 (Believes! ?a2 (the ?a1)))"),

            },
            deletions={
                r("(not (Believes! ?a2 (the ?a1)))"),
                r("(not (Believes! ?a1 (Believes! ?a2 (the ?a1))))"),

            },
        ),




    ]

In [10]:
start =  set(map(r, [
        "(at a p1)",
        "(not (at a p2))",
        "(at b p1)",
        "(not (at b p2))",
        "(at c p1)",
        "(not (at c p2))",

        "(Believes! a (the a))",
        "(Believes! b (the b))",
        "(Believes! c (the c))",

        "(not (Believes! b (the a)))",
        "(not (Believes! c (the a)))",

]))

In [11]:
goal = r("""(and
                (Believes! b (the a))
                (Believes! a (Believes! b (the a)))
                 (not (Believes! c (the a)))
                )""")

## 8. Define the inference schema (DCEC)

In [12]:
logic = read_inference_system(loads("""
{
    :name "DCEC"
    :description "DCEC"

    :schema {

        :R4 {
            :name "R4"
            :inputs [(Knows! ?a ?t ?P)]
            :output [?P]
        }
        :R_K_B {
            :name "Knowledge to belief"
            :inputs [(Knows! ?a ?t ?P)]
            :output (Believes! ?a ?t ?P)
        }
        :Common_to_K1 {
            :name "Common to Knowledge 1"
            :inputs [(Common! ?t ?P)]
            :output (Knows! ?a ?t ?P)

        }
        :Common_to_K2 {
            :name "Common to Knowledge 2 "
            :inputs [(Common! ?t ?P)]
            :output (Knows! ?a t (Knows! ?a ?t ?P))

        }
        :Perception_to_Knowledge {
            :name "Common to Knowledge "
            :inputs [(Perceives! ?a ?t ?P)]
            :output (Knows! ?a ?t ?P)

        }
    }
}

"""))

## 9. Define the prover

In [13]:
def cc_prover(state, test, find_answer=True, max_answers=5):
    return sprover(logic, state, test, find_answer=find_answer, max_answers=max_answers)

## 10. Run Spectra
Solves the epistemic planning problem and returns a plan.

In [14]:
%%time
run_spectra(domain, background, start, goal, actions, cc_prover)

CPU times: user 13 s, sys: 135 ms, total: 13.1 s
Wall time: 14.8 s


([(right a), (right b), (sharesingle a b c p2)],
 frozenset({(Believes! a (Believes! b (the a))),
            (Believes! a (the a)),
            (Believes! b (the a)),
            (Believes! b (the b)),
            (Believes! c (the c)),
            (at a p2),
            (at b p2),
            (at c p1),
            (not (Believes! c (the a))),
            (not (at a p1)),
            (not (at b p1)),
            (not (at c p2))}),
 13,
 [{(Believes! a (the a)),
   (Believes! b (the b)),
   (Believes! c (the c)),
   (at a p1),
   (at b p1),
   (at c p1),
   (not (Believes! b (the a))),
   (not (Believes! c (the a))),
   (not (at a p2)),
   (not (at b p2)),
   (not (at c p2))},
  frozenset({(Believes! a (the a)),
             (Believes! b (the b)),
             (Believes! c (the c)),
             (at a p2),
             (at b p1),
             (at c p1),
             (not (Believes! b (the a))),
             (not (Believes! c (the a))),
             (not (at a p1)),
             (not (